# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"Dataset title: {metadata_json['name']}")
print(f"Description: {metadata_json['description']}")

# Print dataset ID and version
print(f"Identifier (@id): {metadata_json['@id']}")
print(f"Version: {metadata_json.get('version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all available record sets and the corresponding field `@id`s using the Croissant schema.


In [ ]:
# Discover available record sets and fields using their @id
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rset in record_sets:
    print(f"- RecordSet name: {rset.name}")
    print(f"  @id: {rset.id}")
    if hasattr(rset, 'fields') and rset.fields:
        print(f"  Fields:")
        for f in rset.fields:
            print(f"    - {f.name} (id: {f.id}, type: {f.data_type})")
    print()
# For demonstration, preview the first record of each record set
for rset in record_sets:
    print(f"Example record from RecordSet {rset.id}:")
    gen = dataset.records(record_set=rset.id)
    try:
        print(next(gen))
    except StopIteration:
        print("No records found.")


## 3. Data Extraction
Load data from record sets into pandas DataFrames. All `@id`s used below are taken directly from the Croissant schema and previous section.


In [ ]:
# List all available record set @ids
record_set_ids = [rset.id for rset in record_sets]
dataframes = {}

# Load records from each record set into a DataFrame
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print("No records loaded for this record set.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All field and record set references use the entity `@id`.


In [ ]:
# Choose a record set (update the @id to the main protein abundance record set)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df = dataframes.get(main_record_set_id, pd.DataFrame())
    print(f"Main record set: {main_record_set_id}")
    
    # List all fields
    print(f"Columns: {df.columns.tolist()}")
    
    # Guess a numeric field by commonly used protein quantitation fields
    # You may update the field name as needed per dataset schema
    numeric_candidates = [col for col in df.columns if 'abundance' in col.lower() or 'count' in col.lower() or 'coverage' in col.lower() or df[col].dtype in ['int64','float64']]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by categorical field if available
        group_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for automatic analysis.")
else:
    print("No record sets found.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_candidates:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=50, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped, plot mean by group
    if group_field and group_field in grouped_df.index:
        grouped_df[numeric_field].plot(kind='bar', figsize=(10,4), title=f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()


## 6. Conclusion
This notebook demonstrated how to load, explore, and perform basic processing and visualization on a Croissant-described dataset using `mlcroissant`. Key findings from EDA should be further explored with domain-specific analysis relevant to mass spectrometry and protein expression data.

_Notebook complete. You can now use this framework to perform deeper analysis tailored to your research questions._